In [19]:
# ==============================================================================
# BLOCK 1: IMPORTS, MIXED PRECISION SETUP, AND GM CORPUS DOWNLOAD
# ==============================================================================
!pip install -q python-chess requests tqdm

import io
import random
import time
import zipfile
import requests
import chess
import chess.pgn
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Enable cuDNN benchmark for faster convolution kernels
torch.backends.cudnn.benchmark = True

print(f"[INFO] Initializing BOZON v0.5 SE-ResNet + Tactical Value Distillation")
print(f"[INFO] Computing Device: {device}")

PGN_MENTOR_PLAYERS = ["Carlsen", "Kasparov", "Fischer", "Tal", "Karpov", "Nakamura"]

def fetch_pgn_mentor_corpus(players: list) -> str:
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    combined_pgn = []
    for player in players:
        url = f"https://www.pgnmentor.com/players/{player}.zip"
        try:
            r = requests.get(url, headers=headers, timeout=30)
            r.raise_for_status()
            with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
                for fname in zf.namelist():
                    if fname.endswith(".pgn"):
                        text = zf.read(fname).decode("utf-8", errors="ignore")
                        combined_pgn.append(text)
                        print(f"[SUCCESS] Loaded {fname} ({len(text)//1024:,} KB)")
        except Exception as exc:
            print(f"[WARNING] Skipped {player}: {exc}")
    return "\n\n".join(combined_pgn)

pgn_data = fetch_pgn_mentor_corpus(PGN_MENTOR_PLAYERS)

[INFO] Initializing BOZON v0.5 SE-ResNet + Tactical Value Distillation
[INFO] Computing Device: cuda
[SUCCESS] Loaded Carlsen.pgn (5,598 KB)
[SUCCESS] Loaded Kasparov.pgn (1,318 KB)
[SUCCESS] Loaded Fischer.pgn (537 KB)
[SUCCESS] Loaded Tal.pgn (1,421 KB)
[SUCCESS] Loaded Karpov.pgn (2,330 KB)
[SUCCESS] Loaded Nakamura.pgn (7,592 KB)


In [20]:
# ==============================================================================
# BLOCK 2: FAST BITBOARD ENCODING & TACTICAL/CHECKMATE DATA GENERATOR
# ==============================================================================
PIECE_VALUES = {
    chess.PAWN: 1.0,
    chess.KNIGHT: 3.1,
    chess.BISHOP: 3.3,
    chess.ROOK: 5.0,
    chess.QUEEN: 9.5,
    chess.KING: 0.0
}

def fast_board_to_tensor(board: chess.Board) -> np.ndarray:
    """
    Optimized (18, 8, 8) tensor builder using integer bitmasks for speed.
    """
    tensor = np.zeros((18, 8, 8), dtype=np.float32)

    for piece_type in range(1, 7):
        for color in (chess.WHITE, chess.BLACK):
            mask = board.pieces_mask(piece_type, color)
            if not mask:
                continue
            ch = (piece_type - 1) + (0 if color == chess.WHITE else 6)
            while mask:
                lsb = mask & -mask
                sq = lsb.bit_length() - 1
                tensor[ch, 7 - (sq >> 3), sq & 7] = 1.0
                mask ^= lsb

    if board.turn == chess.WHITE:
        tensor[12, :, :] = 1.0

    for move in board.legal_moves:
        to_sq = move.to_square
        tensor[13, 7 - (to_sq >> 3), to_sq & 7] = 1.0

    if bool(board.castling_rights & chess.BB_H1):
        tensor[14, :, :] = 1.0
    if bool(board.castling_rights & chess.BB_A1):
        tensor[15, :, :] = 1.0
    if bool(board.castling_rights & chess.BB_H8):
        tensor[16, :, :] = 1.0
    if bool(board.castling_rights & chess.BB_A8):
        tensor[17, :, :] = 1.0

    return tensor


def encode_move(move: chess.Move) -> int:
    return move.from_square * 64 + move.to_square


def evaluate_board_ground_truth(board: chess.Board) -> float:
    """
    Sharp tactical & material evaluator in [-1.0, +1.0] from ACTIVE player's view.
    Ensures being up a Queen yields ~+0.92 instead of +0.06!
    """
    if board.is_checkmate():
        return -1.0  # Active player is checkmated

    w_mat, b_mat = 0.0, 0.0
    for pt, val in PIECE_VALUES.items():
        if val > 0:
            w_mat += len(board.pieces(pt, chess.WHITE)) * val
            b_mat += len(board.pieces(pt, chess.BLACK)) * val

    diff = (w_mat - b_mat) if board.turn == chess.WHITE else (b_mat - w_mat)
    if board.is_check():
        diff -= 0.4  # Penalty for being in check

    # Sharp scaling: +4 pts (minor piece + pawn) -> tanh(1.0) = +0.76
    # +9 pts (Queen) -> tanh(2.25) = +0.98
    return float(np.tanh(diff / 4.0))


def find_best_tactical_move(board: chess.Board) -> tuple:
    """
    1-Ply Tactical Oracle: Finds checkmate-in-1 or highest material gain move.
    """
    best_move = None
    best_score = -999.0

    for move in board.legal_moves:
        board.push(move)
        if board.is_checkmate():
            board.pop()
            return move, 1.0  # Immediate Checkmate found!

        # Score is negative of opponent's evaluation after our move
        opp_eval = evaluate_board_ground_truth(board)
        score = -opp_eval
        board.pop()

        if score > best_score:
            best_score = score
            best_move = move

    return best_move, best_score

In [21]:
# ==============================================================================
# BLOCK 3: HYBRID DATASET (GM POSITIONS + TACTICAL CHECKMATE/CAPTURE DRILLS)
# ==============================================================================
class BozonDualDataset(Dataset):
    def __init__(self, states: np.ndarray, moves: np.ndarray, values: np.ndarray):
        self.states = torch.from_numpy(states)
        self.moves = torch.from_numpy(moves)
        self.values = torch.from_numpy(values).unsqueeze(1)

    def __len__(self) -> int:
        return len(self.states)

    def __getitem__(self, idx: int):
        return self.states[idx], self.moves[idx], self.values[idx]


def build_v05_hybrid_datasets(pgn_text: str, total_positions: int = 120000):
    """
    Builds a calibrated dataset:
    - 75% Grandmaster positional play (with sharp material-calibrated Value targets)
    - 25% Tactical Drills (Checkmates in 1, Capturing hanging Queens/Rooks, Endgames)
    """
    t0 = time.time()
    train_max = int(total_positions * 0.90)
    val_max = total_positions - train_max

    tr_s = np.zeros((train_max, 18, 8, 8), dtype=np.float32)
    tr_m = np.zeros((train_max,), dtype=np.int64)
    tr_v = np.zeros((train_max,), dtype=np.float32)

    va_s = np.zeros((val_max, 18, 8, 8), dtype=np.float32)
    va_m = np.zeros((val_max,), dtype=np.int64)
    va_v = np.zeros((val_max,), dtype=np.float32)

    pgn_io = io.StringIO(pgn_text)
    game_idx, tr_idx, va_idx = 0, 0, 0

    print(f"[INFO] Building {total_positions:,} Hybrid (GM + Tactical) positions...")

    while tr_idx < train_max or va_idx < val_max:
        game = chess.pgn.read_game(pgn_io)
        if game is None:
            break

        res = game.headers.get("Result", "*")
        if res not in ("1-0", "0-1", "1/2-1/2"):
            continue
        outcome_w = 1.0 if res == "1-0" else (-1.0 if res == "0-1" else 0.0)

        game_idx += 1
        is_val = (game_idx % 10 == 0)
        board = game.board()

        for ply, move in enumerate(game.mainline_moves()):
            # 1. Standard GM Position
            mat_eval = evaluate_board_ground_truth(board)
            out_active = outcome_w if board.turn == chess.WHITE else -outcome_w
            # Weight material truth 75% and distant game result 25% (fixes Value dilution!)
            target_val = 0.75 * mat_eval + 0.25 * out_active

            state_t = fast_board_to_tensor(board)
            move_i = encode_move(move)

            if is_val and va_idx < val_max:
                va_s[va_idx], va_m[va_idx], va_v[va_idx] = state_t, move_i, target_val
                va_idx += 1
            elif not is_val and tr_idx < train_max:
                tr_s[tr_idx], tr_m[tr_idx], tr_v[tr_idx] = state_t, move_i, target_val
                tr_idx += 1

            # 2. Tactical Augmentation (Every 4th move, simulate a blunder by opponent
            #    so BOZON learns to punish blunders, capture free pieces, and deliver mate)
            if ply > 8 and (ply % 4 == 0) and not board.is_game_over():
                legal_list = list(board.legal_moves)
                if len(legal_list) > 3:
                    random_blunder = random.choice(legal_list)
                    board.push(random_blunder)
                    if not board.is_game_over():
                        tac_move, tac_val = find_best_tactical_move(board)
                        if tac_move is not None and abs(tac_val) > 0.35:
                            tac_state = fast_board_to_tensor(board)
                            tac_idx = encode_move(tac_move)
                            if not is_val and tr_idx < train_max:
                                tr_s[tr_idx], tr_m[tr_idx], tr_v[tr_idx] = tac_state, tac_idx, tac_val
                                tr_idx += 1
                    board.pop()

            board.push(move)

            if (tr_idx + va_idx) % 30000 == 0 and (tr_idx + va_idx) > 0:
                print(f"[PROGRESS] Extracted {tr_idx + va_idx:,} / {total_positions:,} ({time.time()-t0:.1f}s)")

            if tr_idx >= train_max and va_idx >= val_max:
                break

    print(f"[SUCCESS] Dataset built in {time.time()-t0:.1f}s | Train: {tr_idx:,} | Val: {va_idx:,}")
    return (
        BozonDualDataset(tr_s[:tr_idx], tr_m[:tr_idx], tr_v[:tr_idx]),
        BozonDualDataset(va_s[:va_idx], va_m[:va_idx], va_v[:va_idx])
    )


train_dataset, val_dataset = build_v05_hybrid_datasets(pgn_data, total_positions=120000)
del pgn_data

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False, pin_memory=True)

[INFO] Building 120,000 Hybrid (GM + Tactical) positions...
[PROGRESS] Extracted 30,000 / 120,000 (10.4s)
[PROGRESS] Extracted 60,000 / 120,000 (20.9s)
[PROGRESS] Extracted 90,000 / 120,000 (31.2s)
[PROGRESS] Extracted 120,000 / 120,000 (46.6s)
[SUCCESS] Dataset built in 46.6s | Train: 108,000 | Val: 12,000


In [22]:
# ==============================================================================
# BLOCK 4: BOZON v0.5 SE-RESNET ARCHITECTURE (CHANNEL ATTENTION)
# ==============================================================================
class SEBlock(nn.Module):
    """
    Squeeze-and-Excitation Attention Block.
    Allows the network to globally attend to long-range piece interactions
    (Rooks, Bishops, Queens, and King safety across all 64 squares).
    """
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scale = self.fc(x).unsqueeze(-1).unsqueeze(-1)
        return x * scale


class SEResidualBlock(nn.Module):
    def __init__(self, channels: int, dropout_p: float = 0.1):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.drop = nn.Dropout2d(p=dropout_p)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.se = SEBlock(channels, reduction=8)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out += residual
        return F.relu(out, inplace=True)


class BozonSEResNet(nn.Module):
    """
    BOZON v0.5 SE-ResNet Dual-Head Architecture (10 SE-Residual Blocks).
    """
    def __init__(self, num_blocks: int = 10, channels: int = 128):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(18, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )

        self.res_tower = nn.Sequential(
            *[SEResidualBlock(channels, dropout_p=0.1) for _ in range(num_blocks)]
        )

        self.policy_head = nn.Sequential(
            nn.Conv2d(channels, 32, kernel_size=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Dropout(p=0.25),
            nn.Linear(32 * 8 * 8, 4096)
        )

        self.value_head = nn.Sequential(
            nn.Conv2d(channels, 16, kernel_size=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(16 * 8 * 8, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(256, 1),
            nn.Tanh()
        )

    def forward(self, x: torch.Tensor):
        features = self.res_tower(self.stem(x))
        return self.policy_head(features), self.value_head(features)


model = BozonSEResNet(num_blocks=10, channels=128).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[INFO] BozonSEResNet (10 SE-Blocks) initialized | Params: {total_params:,}")

[INFO] BozonSEResNet (10 SE-Blocks) initialized | Params: 11,677,793


In [23]:
# ==============================================================================
# BLOCK 5: FAST MIXED-PRECISION (AMP FP16) TRAINING & CALIBRATION TEST
# ==============================================================================
policy_criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
value_criterion = nn.MSELoss()

optimizer = optim.AdamW(model.parameters(), lr=2e-3, weight_decay=3e-4)
epochs = 15
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

def evaluate_v05(model: nn.Module, dataloader: DataLoader):
    model.eval()
    pol_loss_sum, val_mse_sum = 0.0, 0.0
    top1, top3, total = 0, 0, 0
    ce = nn.CrossEntropyLoss()
    mse = nn.MSELoss()

    with torch.no_grad():
        for states, moves, values in dataloader:
            states, moves, values = states.to(device), moves.to(device), values.to(device)
            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                p_logits, v_preds = model(states)
                p_loss = ce(p_logits, moves)
                v_loss = mse(v_preds, values)

            pol_loss_sum += p_loss.item() * states.size(0)
            val_mse_sum += v_loss.item() * states.size(0)

            _, top3_idx = torch.topk(p_logits, k=3, dim=1)
            top1 += (top3_idx[:, 0] == moves).sum().item()
            top3 += (top3_idx == moves.unsqueeze(1)).any(dim=1).sum().item()
            total += states.size(0)

    return pol_loss_sum / total, val_mse_sum / total, (top1 / total) * 100, (top3 / total) * 100


best_score = -999.0
weights_path = "bozon_v0.5_seresnet.pt"

print("[INFO] Starting High-Speed AMP FP16 Training...")

for epoch in range(1, epochs + 1):
    t0 = time.time()
    model.train()
    tr_p, tr_v, total_tr = 0.0, 0.0, 0

    for states, moves, values in train_loader:
        states, moves, values = states.to(device), moves.to(device), values.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            p_logits, v_preds = model(states)
            loss_p = policy_criterion(p_logits, moves)
            loss_v = value_criterion(v_preds, values)
            loss = loss_p + (2.5 * loss_v)  # Stronger emphasis on accurate Value Head

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        tr_p += loss_p.item() * states.size(0)
        tr_v += loss_v.item() * states.size(0)
        total_tr += states.size(0)

    scheduler.step()
    val_p, val_mse, val_top1, val_top3 = evaluate_v05(model, val_loader)
    elapsed = time.time() - t0

    # Combined quality score: Higher Top-1 and Lower Value MSE
    composite_score = val_top1 - (val_mse * 50.0)
    saved_str = ""
    if composite_score > best_score:
        best_score = composite_score
        torch.save(model.state_dict(), weights_path)
        saved_str = " [BEST SAVED]"

    print(
        f"[EPOCH {epoch:02d}/{epochs:02d}] ({elapsed:.1f}s) | "
        f"TrPol: {tr_p/total_tr:.3f} | ValPol: {val_p:.3f} | "
        f"ValMSE: {val_mse:.4f} | Top-1: {val_top1:.2f}% | Top-3: {val_top3:.2f}%{saved_str}"
    )

# Reload best checkpoint and run calibration tests
model.load_state_dict(torch.load(weights_path, map_location=device))
model.eval()
checksum = float(next(model.parameters()).sum().item())
print(f"\n[SUCCESS] Saved '{weights_path}' | Checksum: {checksum:.6f}")

def quick_eval(fen: str, title: str):
    b = chess.Board(fen)
    t = torch.tensor(fast_board_to_tensor(b), dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        _, v = model(t)
    w_eval = v.item() if b.turn == chess.WHITE else -v.item()
    w_prob = (w_eval + 1.0) * 50.0
    print(f"[{title}] Value Eval: {w_eval:+.3f} | White Win Chance: {w_prob:.1f}%")

quick_eval(chess.STARTING_FEN, "TEST 1 - Starting Board")
quick_eval("rnb1kbnr/pppp1ppp/8/4p3/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2", "TEST 2 - White Up a Queen")
quick_eval("rnbqkbnr/pppp1ppp/8/4p3/6P1/5P2/PPPPP2P/RNBQKBNR b KQkq - 0 2", "TEST 3 - Fool's Mate Opportunity (Black to Mate)")

[INFO] Starting High-Speed AMP FP16 Training...
[EPOCH 01/15] (15.9s) | TrPol: 5.569 | ValPol: 3.889 | ValMSE: 0.1045 | Top-1: 18.88% | Top-3: 34.21% [BEST SAVED]
[EPOCH 02/15] (15.3s) | TrPol: 3.472 | ValPol: 3.329 | ValMSE: 0.1009 | Top-1: 24.54% | Top-3: 42.98% [BEST SAVED]
[EPOCH 03/15] (15.3s) | TrPol: 2.757 | ValPol: 3.239 | ValMSE: 0.1029 | Top-1: 26.03% | Top-3: 44.18% [BEST SAVED]
[EPOCH 04/15] (15.4s) | TrPol: 2.325 | ValPol: 3.312 | ValMSE: 0.0868 | Top-1: 26.25% | Top-3: 44.90% [BEST SAVED]
[EPOCH 05/15] (15.2s) | TrPol: 2.019 | ValPol: 3.381 | ValMSE: 0.0856 | Top-1: 26.88% | Top-3: 45.64% [BEST SAVED]
[EPOCH 06/15] (15.1s) | TrPol: 1.801 | ValPol: 3.474 | ValMSE: 0.0870 | Top-1: 27.42% | Top-3: 46.00% [BEST SAVED]
[EPOCH 07/15] (15.3s) | TrPol: 1.632 | ValPol: 3.604 | ValMSE: 0.0845 | Top-1: 27.41% | Top-3: 45.73% [BEST SAVED]
[EPOCH 08/15] (15.2s) | TrPol: 1.494 | ValPol: 3.665 | ValMSE: 0.0741 | Top-1: 27.35% | Top-3: 45.59% [BEST SAVED]
[EPOCH 09/15] (15.4s) | TrPol: 1